<a href="https://colab.research.google.com/github/vishal786-commits/machine-learning-journey/blob/main/classical-ml/random-forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ensemble Methods in Machine Learning & Random Forests**
### A Rigorous, Intuition-First Treatment

---

**Topics Covered**

1. Why Single Models Fall Short — Bias-Variance Decomposition  
2. The Core Idea Behind Ensembles  
3. Bootstrap Sampling  
4. Bagging (Bootstrap Aggregating)  
5. Random Forest — Algorithm, Mechanics, and Intuition  
6. Out-of-Bag (OOB) Error  
7. Feature Importance  
8. Boosting — AdaBoost and Gradient Boosting  
9. Stacking  
10. Practical Comparison and Hyperparameter Guide  

---

> **Notation convention:** Throughout this notebook, $\hat{f}$ denotes a learned model, $f^*$ the true function, $x$ an input vector, $y$ the target, and $\mathcal{D}$ the training dataset.

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# ── Sklearn ──────────────────────────────────────────────────────────────────
from sklearn.datasets import (
    load_breast_cancer, load_iris, fetch_california_housing, make_classification
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, RandomForestRegressor,
    AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier,
    GradientBoostingRegressor
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, train_test_split, learning_curve
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from IPython.display import HTML

np.random.seed(42)
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})
print("Setup complete.")

---

## 1. Why Single Models Fall Short — The Bias-Variance Decomposition

### 1.1 The Decomposition

For a regression model under squared-error loss, the expected prediction error at a point $x$ decomposes as:

$$
\underbrace{\mathbb{E}\left[(y - \hat{f}(x))^2\right]}_{\text{Total Error}}
= \underbrace{\left(f^*(x) - \mathbb{E}[\hat{f}(x)]\right)^2}_{\text{Bias}^2}
+ \underbrace{\mathbb{E}\left[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2\right]}_{\text{Variance}}
+ \underbrace{\sigma^2_{\epsilon}}_{\text{Irreducible Noise}}
$$

**Intuition:**
- **Bias** measures how wrong the average prediction is — systematic error.  
  *A linear model fit to a sinusoidal pattern has high bias.*
- **Variance** measures how much predictions swing across different training sets — sensitivity to data.  
  *A deep decision tree memorises training noise; swap the dataset slightly and predictions change drastically.*
- **Irreducible noise** ($\sigma^2_\epsilon$) is the floor — no model can beat it.

### 1.2 The Fundamental Trade-off

| Model Complexity | Bias | Variance | Behaviour |
|---|---|---|---|
| Low (shallow tree, linear model) | High | Low | Underfits |
| High (deep tree, deep network) | Low | High | Overfits |

The goal of ensemble methods is to **break this trade-off** — specifically:
- **Bagging / Random Forest**: keeps bias low by using complex base learners, then *averages out* variance.
- **Boosting**: starts from high-bias weak learners and *sequentially reduces* bias.

In [ ]:
# ── Animate Bias-Variance: single deep tree vs. ensemble average ─────────────
# True function: sin curve. Train 30 datasets; for each fit one deep tree.
# Watch individual predictions swing vs. how the mean stabilises.

def true_fn(x):
    return np.sin(2 * np.pi * x)

X_plot = np.linspace(0, 1, 300)
n_datasets = 30
preds = []

for _ in range(n_datasets):
    X_tr = np.random.uniform(0, 1, 25).reshape(-1, 1)
    y_tr = true_fn(X_tr.ravel()) + np.random.normal(0, 0.3, 25)
    tree = DecisionTreeRegressor(max_depth=8)
    tree.fit(X_tr, y_tr)
    preds.append(tree.predict(X_plot.reshape(-1, 1)))

preds = np.array(preds)           # (30, 300)
cumulative_mean = np.cumsum(preds, axis=0) / np.arange(1, n_datasets + 1)[:, None]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax in axes:
    ax.plot(X_plot, true_fn(X_plot), 'k--', lw=2, label='True function', zorder=5)
    ax.set_xlim(0, 1); ax.set_ylim(-2, 2)

line_single, = axes[0].plot([], [], color='steelblue', alpha=0.8, lw=1.5)
line_mean,   = axes[1].plot([], [], color='firebrick', lw=2.2)

axes[0].set_title('Single Deep Tree\n(high variance — changes with data)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
axes[0].legend(loc='upper right', fontsize=8)

axes[1].set_title('Ensemble Mean\n(variance collapses as trees accumulate)')
axes[1].set_xlabel('x')

title_text = axes[0].text(0.02, 1.75, '', fontsize=9)

def animate(i):
    line_single.set_data(X_plot, preds[i])
    line_mean.set_data(X_plot, cumulative_mean[i])
    title_text.set_text(f'Tree {i+1} / {n_datasets}')
    return line_single, line_mean, title_text

ani = animation.FuncAnimation(fig, animate, frames=n_datasets, interval=220, blit=True)
plt.tight_layout()
HTML(ani.to_jshtml())

**Reading the animation:**  
The left panel shows an individual deep tree prediction changing wildly across training sets — this is high variance.  
The right panel accumulates the mean: after ~10 trees the average converges to the true sinusoid. Variance has been averaged away; bias (determined by the tree family) is unchanged.

---

## 2. The Core Idea Behind Ensembles

### 2.1 Wisdom of Crowds

**Real-world analogy:** In 1907, Francis Galton observed that the median of 800 independent crowd guesses for an ox's weight (1,207 lbs) was virtually identical to the true value (1,198 lbs). No single expert was that accurate.

The mathematical foundation is simple. Suppose we have $M$ independent models, each with variance $\sigma^2$ and zero correlation:

$$\text{Var}\left(\frac{1}{M}\sum_{m=1}^M \hat{f}_m(x)\right) = \frac{\sigma^2}{M}$$

Averaging $M$ independent models divides variance by $M$. In practice models are **correlated** (same data), so:

$$\text{Var}(\bar{\hat{f}}) = \rho \sigma^2 + \frac{1-\rho}{M} \sigma^2$$

where $\rho$ is the pairwise correlation between model predictions. As $M \to \infty$:

$$\text{Var}(\bar{\hat{f}}) \to \rho \sigma^2$$

**Key implication:** The limit of variance reduction is set by correlation, not by adding more trees. This is precisely why Random Forest introduces **feature randomness** — to *de-correlate* the trees.

### 2.2 Three Ensemble Paradigms

| Paradigm | Base Learners | How Combined | Targets |
|---|---|---|---|
| **Bagging** | Parallel, independent | Average / majority vote | Variance reduction |
| **Boosting** | Sequential, each corrects prior | Weighted sum | Bias reduction |
| **Stacking** | Diverse parallel models | Meta-learner | Both |

---

## 3. Bootstrap Sampling

Bootstrap is the statistical engine behind Bagging. Given a dataset $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^n$, a **bootstrap sample** $\mathcal{D}^*$ is drawn by sampling $n$ points **with replacement**.

**Expected fraction included:** Each observation has probability $\left(1 - \frac{1}{n}\right)^n \to e^{-1} \approx 36.8\%$ of being excluded. So approximately **63.2% of unique samples** appear in each bootstrap replicate.

The excluded samples are the **Out-of-Bag (OOB)** set — a free validation set we will exploit later.

In [ ]:
# ── Animate Bootstrap Sampling ───────────────────────────────────────────────
n_samples = 12
n_bootstrap = 6
orig = np.arange(n_samples)

bootstrap_draws = [
    np.random.choice(orig, size=n_samples, replace=True)
    for _ in range(n_bootstrap)
]

fig, ax = plt.subplots(figsize=(11, 4))
ax.set_xlim(-0.5, n_samples - 0.5)
ax.set_ylim(-1.5, 2.5)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title('Bootstrap Sampling: original 12 samples → bootstrap replicate')

# Draw original row
orig_circles = []
for i, idx in enumerate(orig):
    c = plt.Circle((i, 2), 0.35, color='steelblue', zorder=3)
    ax.add_patch(c)
    ax.text(i, 2, str(idx), ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    orig_circles.append(c)

ax.text(-0.5, 2, 'Original $\mathcal{D}$:', ha='right', va='center', fontsize=9, fontstyle='italic')
ax.text(-0.5, 0.3, 'Bootstrap $\mathcal{D}^*$:', ha='right', va='center', fontsize=9, fontstyle='italic')
ax.text(-0.5, -1, 'OOB set:', ha='right', va='center', fontsize=9, color='firebrick', fontstyle='italic')

bt_patches = []
oob_patches = []
info_text = ax.text(5.5, -1.5, '', ha='center', fontsize=9)

def init():
    for p in bt_patches + oob_patches:
        p.remove()
    bt_patches.clear(); oob_patches.clear()
    info_text.set_text('')
    return []

def animate_bs(frame):
    for p in bt_patches + oob_patches:
        p.remove()
    bt_patches.clear(); oob_patches.clear()

    draw = bootstrap_draws[frame]
    counts = np.bincount(draw, minlength=n_samples)
    oob = np.where(counts == 0)[0]
    pct_unique = (counts > 0).sum() / n_samples * 100

    for pos, idx in enumerate(draw):
        color = 'seagreen' if counts[idx] == 1 else 'darkorange'
        c = plt.Circle((pos, 0.3), 0.35, color=color, zorder=3)
        ax.add_patch(c)
        ax.text(pos, 0.3, str(idx), ha='center', va='center',
                fontsize=8, color='white', fontweight='bold')
        bt_patches.append(c)

    for pos, idx in enumerate(oob):
        c = plt.Circle((pos, -1.0), 0.35, color='firebrick', alpha=0.7, zorder=3)
        ax.add_patch(c)
        ax.text(pos, -1.0, str(idx), ha='center', va='center',
                fontsize=8, color='white', fontweight='bold')
        oob_patches.append(c)

    info_text.set_text(
        f'Bootstrap {frame+1}/{n_bootstrap} | Unique: {pct_unique:.0f}% | '
        f'OOB count: {len(oob)} ({len(oob)/n_samples*100:.0f}%)'
    )
    return bt_patches + oob_patches + [info_text]

ani2 = animation.FuncAnimation(fig, animate_bs, init_func=init,
                                frames=n_bootstrap, interval=1200, blit=False)
plt.tight_layout()
HTML(ani2.to_jshtml())

**Orange circles** indicate samples drawn more than once (duplicates). Red circles are OOB — excluded from this bootstrap replicate but available for unbiased validation.

---

## 4. Bagging (Bootstrap Aggregating)

**Algorithm** (Breiman, 1996):

1. For $m = 1, \ldots, M$:
   - Draw bootstrap sample $\mathcal{D}^{(m)}$ from $\mathcal{D}$
   - Train base model $\hat{f}^{(m)}$ on $\mathcal{D}^{(m)}$
2. Aggregate:
   - **Regression:** $\hat{f}_{\text{bag}}(x) = \frac{1}{M}\sum_{m=1}^M \hat{f}^{(m)}(x)$
   - **Classification:** $\hat{y}_{\text{bag}}(x) = \text{mode}\left\{\hat{f}^{(m)}(x)\right\}_{m=1}^M$

**Why does it work?**  
Averaging unbiased estimators preserves bias while reducing variance by de-coupling predictions across training sets. The bootstrap induces diversity — each model is trained on a slightly different data view.

**Bagging works best with:** high-variance, low-bias learners (e.g., fully-grown decision trees). Bagging a low-variance model (linear regression) gives minimal gain.

In [ ]:
# ── Bagging: Single tree vs. Bagged trees on Breast Cancer ───────────────────
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.25, random_state=42)

n_tree_range = [1, 5, 10, 25, 50, 100, 200]
single_scores, bag_scores = [], []

for n in n_tree_range:
    # Single tree (repeated n times — pick the best)
    st = DecisionTreeClassifier(max_depth=None)
    st.fit(X_tr, y_tr)
    single_scores.append(accuracy_score(y_te, st.predict(X_te)))

    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=None),
        n_estimators=n, random_state=42, n_jobs=-1
    )
    bag.fit(X_tr, y_tr)
    bag_scores.append(accuracy_score(y_te, bag.predict(X_te)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.axhline(single_scores[0], color='steelblue', lw=1.5, ls='--', label=f'Single tree ({single_scores[0]:.3f})')
ax.plot(n_tree_range, bag_scores, 'o-', color='firebrick', lw=2, label='Bagged trees')
ax.set_xlabel('Number of estimators')
ax.set_ylabel('Test Accuracy')
ax.set_title('Bagging vs. Single Decision Tree — Breast Cancer Dataset')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Single deep tree accuracy : {single_scores[0]:.4f}")
print(f"Bagged (200 trees) accuracy: {bag_scores[-1]:.4f}")

---

## 5. Random Forest

### 5.1 From Bagging to Random Forest

Bagging reduces variance, but the trees remain **correlated** because they all have access to the same features at each split. If there is one very strong predictor, every tree will use it near the root, producing near-identical structures.

**Random Forest** (Breiman, 2001) adds a second source of randomness:

> At each node split, instead of searching all $p$ features, only a **random subset of $m_{\text{try}}$ features** is considered.

This *feature subsampling* forces different trees to discover different patterns, reducing $\rho$ in the variance formula and unlocking further variance reduction beyond bagging's limit.

### 5.2 The Algorithm

```
Input: Training data D, number of trees M, features per split m_try

For m = 1 to M:
    1. Draw bootstrap sample D*(m) from D
    2. Grow tree T(m) on D*(m):
       While node is not pure or min_size reached:
           a. Sample m_try features at random (no replacement)
           b. Find best split among sampled features
           c. Split node; recurse on children
    3. Store T(m) (no pruning)

Prediction:
    Regression:      f_RF(x) = (1/M) * sum_m T(m)(x)
    Classification:  f_RF(x) = majority_vote { T(m)(x) }
```

### 5.3 Default Hyperparameter Choices

| Parameter | Classification default | Regression default | Effect |
|---|---|---|---|
| `max_features` (`m_try`) | $\lfloor\sqrt{p}\rfloor$ | $\lfloor p/3 \rfloor$ | Controls tree correlation |
| `n_estimators` | 100 | 100 | More = better (diminishing returns) |
| `max_depth` | None (fully grown) | None | Keeps bias low |
| `min_samples_leaf` | 1 | 1 | Slight increase improves regression |

**The $\sqrt{p}$ rule** is empirically robust: small enough to de-correlate trees, large enough to find informative splits.

In [ ]:
# ── Visualise a single tree from the forest (shallow for readability) ────────
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

rf_iris = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
rf_iris.fit(X_iris, y_iris)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, ax in enumerate(axes):
    plot_tree(
        rf_iris.estimators_[i],
        feature_names=iris.feature_names,
        class_names=iris.target_names,
        filled=True, rounded=True,
        ax=ax, fontsize=7
    )
    ax.set_title(f'Tree {i+1} from Random Forest\n(trained on different bootstrap + feature subset)')

plt.suptitle(
    'Three trees in the same forest — note different root features despite same dataset',
    y=1.01, fontsize=11
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Decision boundary: Single tree vs. Random Forest ────────────────────────
# Use 2 features from Iris for visualisation clarity
X2 = X_iris[:, 2:4]   # petal length, petal width
y2 = y_iris

xx, yy = np.meshgrid(
    np.linspace(X2[:, 0].min() - 0.3, X2[:, 0].max() + 0.3, 400),
    np.linspace(X2[:, 1].min() - 0.3, X2[:, 1].max() + 0.3, 400)
)
grid = np.c_[xx.ravel(), yy.ravel()]

models = {
    'Single Decision Tree\n(max_depth=None)': DecisionTreeClassifier(max_depth=None, random_state=42),
    'Random Forest\n(100 trees, max_depth=None)': RandomForestClassifier(n_estimators=100, random_state=42)
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cmap_back = plt.cm.RdYlBu
colors = ['#e74c3c', '#2ecc71', '#3498db']

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X2, y2)
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=cmap_back)
    for cls, color in zip([0, 1, 2], colors):
        mask = y2 == cls
        ax.scatter(X2[mask, 0], X2[mask, 1], c=color, s=30, edgecolors='white',
                   lw=0.5, label=iris.target_names[cls])
    ax.set_xlabel('Petal Length (cm)')
    ax.set_ylabel('Petal Width (cm)')
    ax.set_title(name)
    ax.legend(fontsize=8)

plt.suptitle('Decision Boundaries: Single Tree vs. Random Forest\n'
             'Single tree produces jagged, overfit boundaries; RF produces smoother, more generalisable regions',
             y=1.02, fontsize=10)
plt.tight_layout()
plt.show()

### 5.4 Effect of `n_estimators` and `max_features` on Error

In [ ]:
# ── OOB error vs. n_estimators (tracked during fit) ─────────────────────────
X_bc, y_bc = load_breast_cancer(return_X_y=True)

rf_oob = RandomForestClassifier(
    n_estimators=200, oob_score=True,
    warm_start=True, random_state=42
)

n_range = range(1, 201, 5)
oob_errors = []

for n in n_range:
    rf_oob.set_params(n_estimators=n)
    rf_oob.fit(X_bc, y_bc)
    oob_errors.append(1 - rf_oob.oob_score_)

# ── Effect of max_features ───────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.25, random_state=42)
p = X_tr.shape[1]
mf_options = [1, 2, 3, 5, 8, 10, 15, p]
mf_labels   = [str(m) if m < p else 'all' for m in mf_options]
mf_scores   = []

for mf in mf_options:
    rf = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=42)
    rf.fit(X_tr, y_tr)
    mf_scores.append(accuracy_score(y_te, rf.predict(X_te)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(list(n_range), oob_errors, color='steelblue', lw=2)
ax1.axvline(x=50, color='firebrick', ls='--', lw=1, label='~50 trees (elbow)')
ax1.set_xlabel('Number of Trees')
ax1.set_ylabel('OOB Error Rate')
ax1.set_title('OOB Error vs. n_estimators\n(Breast Cancer — no separate test set needed)')
ax1.legend()

ax2.bar(mf_labels, mf_scores, color='steelblue', edgecolor='white')
ax2.axvline(x=mf_labels.index(str(int(np.sqrt(p)))), color='firebrick',
            ls='--', lw=1.5, label=f'sqrt(p) = {int(np.sqrt(p))}')
ax2.set_xlabel('max_features (m_try)')
ax2.set_ylabel('Test Accuracy')
ax2.set_title('Effect of max_features\n(default sqrt(p) is near-optimal)')
ax2.legend()
ax2.set_ylim(0.93, 0.99)

plt.tight_layout()
plt.show()

---

## 6. Out-of-Bag (OOB) Error

Because each tree is trained on a bootstrap sample (~63.2% of the data), each observation $x_i$ is **out-of-bag** for roughly 37% of the trees. We can therefore evaluate $x_i$ using only those trees that never saw it during training:

$$\hat{f}^{\text{OOB}}(x_i) = \frac{1}{|\{m : i \in \text{OOB}^{(m)}\}|} \sum_{m: i \in \text{OOB}^{(m)}} \hat{f}^{(m)}(x_i)$$

The OOB error computed this way is an **approximately unbiased estimator** of the generalisation error, eliminating the need for a separate validation split.

**When to rely on OOB error:** When data is scarce and you cannot afford a held-out set. In general, cross-validation remains preferred for final model selection.

In [ ]:
# ── Compare OOB error vs. cross-validation error ─────────────────────────────
rf_full = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42)
cv_scores = cross_val_score(rf_full, X_bc, y_bc, cv=5, scoring='accuracy')
rf_full.fit(X_bc, y_bc)
oob_acc = rf_full.oob_score_

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['OOB Accuracy', '5-Fold CV Mean'],
       [oob_acc, cv_scores.mean()],
       color=['steelblue', 'seagreen'], edgecolor='white', width=0.4)
ax.errorbar(['5-Fold CV Mean'], [cv_scores.mean()],
            yerr=cv_scores.std(), fmt='none', color='black', capsize=5)
ax.set_ylim(0.95, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title(f'OOB ({oob_acc:.4f}) ≈ 5-Fold CV ({cv_scores.mean():.4f} ± {cv_scores.std():.4f})\n'
             'OOB is a reliable free estimate of generalisation error')
plt.tight_layout()
plt.show()

---

## 7. Feature Importance

Random Forest provides two measures of feature importance:

### 7.1 Mean Decrease in Impurity (MDI) — Gini Importance

For each feature $j$, sum the weighted impurity reduction at every node in every tree where $j$ was the split variable:

$$\text{MDI}(j) = \frac{1}{M} \sum_{m=1}^M \sum_{t \in T_m : \text{split on } j} n_t \cdot \Delta I(t)$$

where $n_t$ is the fraction of samples at node $t$ and $\Delta I(t)$ is the impurity reduction.

**Limitation:** MDI is **biased toward high-cardinality features** (features with many unique values receive artificially high importance). It also cannot be interpreted on unseen data.

### 7.2 Permutation Importance

Shuffle feature $j$ in the test set (breaking its relationship to $y$) and measure the increase in error:

$$\text{PI}(j) = \frac{1}{K} \sum_{k=1}^K \left[ \text{err}(X_{j}^{\text{perm}_k}) - \text{err}(X) \right]$$

Permutation importance is **unbiased**, computed on test data, and generalises to any model. It is the recommended method for reliable feature attribution.

In [ ]:
# ── MDI vs. Permutation Importance on Breast Cancer ─────────────────────────
rf_fi = RandomForestClassifier(n_estimators=200, random_state=42)
rf_fi.fit(X_tr, y_tr)

mdi_imp = pd.Series(rf_fi.feature_importances_, index=load_breast_cancer().feature_names)
mdi_top = mdi_imp.nlargest(15)

perm_result = permutation_importance(rf_fi, X_te, y_te, n_repeats=30, random_state=42, n_jobs=-1)
perm_imp = pd.Series(perm_result.importances_mean, index=load_breast_cancer().feature_names)
perm_top = perm_imp.nlargest(15)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

mdi_top.sort_values().plot(kind='barh', ax=ax1, color='steelblue', edgecolor='white')
ax1.set_title('MDI (Gini Importance)\nBiased toward high-cardinality features')
ax1.set_xlabel('Mean Impurity Decrease')

perm_sorted = perm_top.sort_values()
perm_err = pd.Series(
    perm_result.importances_std[perm_top.index.map(lambda x: list(load_breast_cancer().feature_names).index(x))],
    index=perm_sorted.index
)
ax2.barh(range(len(perm_sorted)), perm_sorted.values, color='seagreen', edgecolor='white')
ax2.set_yticks(range(len(perm_sorted)))
ax2.set_yticklabels(perm_sorted.index, fontsize=8)
ax2.set_title('Permutation Importance (test set)\nUnbiased, model-agnostic')
ax2.set_xlabel('Mean Accuracy Decrease when Shuffled')

plt.suptitle('Feature Importance Comparison — Breast Cancer Dataset', y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

---

## 8. Boosting

Boosting operates on a fundamentally different principle from bagging. Rather than reducing variance through parallelism, it **sequentially reduces bias** by having each new model focus on the mistakes of the previous ensemble.

### 8.1 AdaBoost (Adaptive Boosting)

**Core idea:** Maintain a weight distribution over training samples. After each round, increase weights for misclassified examples so the next learner pays more attention to hard cases.

**Algorithm:**
1. Initialise weights: $w_i^{(1)} = 1/n$
2. For $m = 1, \ldots, M$:
   - Train $\hat{f}^{(m)}$ on weighted data
   - Compute weighted error: $\epsilon_m = \sum_{i: \hat{f}^{(m)}(x_i) \neq y_i} w_i$
   - Learner weight: $\alpha_m = \frac{1}{2} \ln\frac{1-\epsilon_m}{\epsilon_m}$ (strong learners get higher weight)
   - Update sample weights: $w_i^{(m+1)} \propto w_i^{(m)} \cdot e^{-\alpha_m y_i \hat{f}^{(m)}(x_i)}$
3. Final: $\hat{F}(x) = \text{sign}\left(\sum_m \alpha_m \hat{f}^{(m)}(x)\right)$

### 8.2 Gradient Boosting

Gradient boosting generalises AdaBoost by viewing boosting as **gradient descent in function space**.

At each step, fit the new tree to the **negative gradient of the loss** (pseudo-residuals):

$$r_i^{(m)} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F^{(m-1)}}$$

For MSE loss, $L = \frac{1}{2}(y - F)^2$, so $r_i = y_i - F^{(m-1)}(x_i)$ — the residuals.  
For log-loss, the pseudo-residuals are the difference between actual and predicted probabilities.

The ensemble is updated as:
$$F^{(m)}(x) = F^{(m-1)}(x) + \eta \cdot h^{(m)}(x)$$

where $\eta$ is the learning rate (shrinkage) and $h^{(m)}$ is the new tree fit to $\{r_i^{(m)}\}$.

**Intuition via residuals (regression):**
> Fit the data → compute what you got wrong (residuals) → fit those residuals → add correction → repeat.
> It is as if each tree patches the previous model's blind spots.

In [ ]:
# ── Animate Gradient Boosting: residuals shrinking with each stage ────────────
np.random.seed(7)
X_gb = np.sort(np.random.uniform(0, 5, 80))
y_gb = np.sin(X_gb) + 0.5 * np.cos(2 * X_gb) + np.random.normal(0, 0.15, 80)

X_gb_r = X_gb.reshape(-1, 1)
X_fine = np.linspace(0, 5, 400).reshape(-1, 1)

n_stages = [1, 2, 5, 10, 25, 50, 100]
stage_preds = []
for n in n_stages:
    gb = GradientBoostingRegressor(n_estimators=n, max_depth=2, learning_rate=0.3, random_state=42)
    gb.fit(X_gb_r, y_gb)
    stage_preds.append(gb.predict(X_fine))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
true_curve = np.sin(X_fine.ravel()) + 0.5 * np.cos(2 * X_fine.ravel())

ax_fit, ax_res = axes
ax_fit.scatter(X_gb, y_gb, s=15, alpha=0.5, color='grey', label='data')
ax_fit.plot(X_fine, true_curve, 'k--', lw=1.5, label='true')
line_fit, = ax_fit.plot([], [], color='firebrick', lw=2)
ax_fit.set_xlim(0, 5); ax_fit.set_ylim(-2.2, 2.2)
ax_fit.set_title('Gradient Boosting Fit'); ax_fit.legend(fontsize=8)

ax_res.axhline(0, color='black', lw=0.8)
res_scatter = ax_res.scatter([], [], s=20, color='steelblue', alpha=0.7)
ax_res.set_xlim(0, 5); ax_res.set_ylim(-1.5, 1.5)
ax_res.set_title('Residuals — shrinking with each stage')
ax_res.set_xlabel('x'); ax_res.set_ylabel('residual')
stage_label = ax_fit.text(0.1, -2.0, '', fontsize=10)

def animate_gb(i):
    line_fit.set_data(X_fine.ravel(), stage_preds[i])
    gb_i = GradientBoostingRegressor(
        n_estimators=n_stages[i], max_depth=2, learning_rate=0.3, random_state=42)
    gb_i.fit(X_gb_r, y_gb)
    res = y_gb - gb_i.predict(X_gb_r)
    res_scatter.set_offsets(np.c_[X_gb, res])
    stage_label.set_text(f'Stages: {n_stages[i]}')
    return line_fit, res_scatter, stage_label

ani3 = animation.FuncAnimation(fig, animate_gb, frames=len(n_stages), interval=900, blit=True)
plt.tight_layout()
HTML(ani3.to_jshtml())

**Key observation:** Residuals collapse toward zero as stages accumulate. Early stages capture gross structure; later stages correct fine-grained errors. Note that with too many stages and a high learning rate, boosting *overfits* — the residuals become noise — unlike Random Forest, which is more robust to over-adding trees.

In [ ]:
# ── AdaBoost vs. Gradient Boosting vs. Random Forest on Breast Cancer ────────
models_cmp = {
    'AdaBoost\n(100 stumps)': AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42),
    'Gradient Boosting\n(100 trees, lr=0.1)': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    'Random Forest\n(100 trees)': RandomForestClassifier(n_estimators=100, random_state=42),
    'Single Decision Tree': DecisionTreeClassifier(random_state=42),
}

results = {}
for name, model in models_cmp.items():
    scores = cross_val_score(model, X_bc, y_bc, cv=5, scoring='accuracy')
    results[name] = (scores.mean(), scores.std())

names, means, stds = zip(*[(k, v[0], v[1]) for k, v in results.items()])

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#95a5a6']
bars = ax.barh(names, means, xerr=stds, color=colors, edgecolor='white', capsize=5, height=0.5)
ax.set_xlabel('5-Fold CV Accuracy')
ax.set_title('Ensemble Methods vs. Single Tree — Breast Cancer (5-Fold CV)')
ax.set_xlim(0.90, 1.00)
for bar, mean, std in zip(bars, means, stds):
    ax.text(mean + std + 0.001, bar.get_y() + bar.get_height()/2,
            f'{mean:.4f} ± {std:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

---

## 9. Stacking (Stacked Generalisation)

### 9.1 Concept

Stacking uses **diverse base learners** to generate a meta-feature matrix, then trains a **meta-learner** on those features.

**Why diversity matters:** If all base learners make identical errors, the meta-learner cannot correct them. The benefit comes from learners that are *wrong in different ways*.

**Training procedure** (to avoid leakage):
1. Split training data into $K$ folds
2. For each fold $k$: train each base learner on the other $K-1$ folds, predict on fold $k$
3. Collect out-of-fold predictions → this is the meta-feature matrix $Z \in \mathbb{R}^{n \times L}$ ($L$ = number of base learners)
4. Train meta-learner on $Z$ to predict $y$
5. For inference: pass each test point through all base learners to form $z$, then through the meta-learner

**Meta-learner choice:** Logistic Regression / Ridge is usually preferred — simple, regularised, interpretable. A complex meta-learner risks overfitting the meta-features.

In [ ]:
# ── Stacking example on Breast Cancer ────────────────────────────────────────
base_learners = [
    ('rf',  RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb',  GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('svm', SVC(probability=True, random_state=42)),
]
meta_learner = LogisticRegression(max_iter=1000)

stacker = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5, passthrough=False
)

stack_scores = cross_val_score(stacker, X_bc, y_bc, cv=5, scoring='accuracy')
rf_scores    = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                               X_bc, y_bc, cv=5, scoring='accuracy')

print(f"Stacking (RF + GB + SVM → LR):  {stack_scores.mean():.4f} ± {stack_scores.std():.4f}")
print(f"Random Forest alone:             {rf_scores.mean():.4f} ± {rf_scores.std():.4f}")
print("\nNote: Stacking gains depend on base learner diversity.")
print("On simple/clean datasets the margin is often small;")
print("gains become more pronounced on harder, noisier problems.")

---

## 10. Practical Comparison and Hyperparameter Guide

### 10.1 Learning Curves — Diagnosing Fit

In [ ]:
# ── Learning curves: RF vs. single tree ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (model, title) in zip(axes, [
    (DecisionTreeClassifier(), 'Single Decision Tree'),
    (RandomForestClassifier(n_estimators=100, random_state=42), 'Random Forest (100 trees)')
]):
    train_sz, train_sc, val_sc = learning_curve(
        model, X_bc, y_bc,
        cv=5, scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 10),
        n_jobs=-1
    )
    ax.plot(train_sz, train_sc.mean(1), 'o-', label='Train', color='steelblue')
    ax.fill_between(train_sz,
                    train_sc.mean(1) - train_sc.std(1),
                    train_sc.mean(1) + train_sc.std(1), alpha=0.15, color='steelblue')
    ax.plot(train_sz, val_sc.mean(1), 'o-', label='Validation', color='firebrick')
    ax.fill_between(train_sz,
                    val_sc.mean(1) - val_sc.std(1),
                    val_sc.mean(1) + val_sc.std(1), alpha=0.15, color='firebrick')
    ax.set_title(title)
    ax.set_xlabel('Training Samples')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0.80, 1.01)
    ax.legend()

plt.suptitle('Learning Curves: Single Tree (high gap = overfitting) vs. Random Forest (low gap)',
             y=1.02, fontsize=10)
plt.tight_layout()
plt.show()

### 10.2 Full Comparison — California Housing (Regression)

In [ ]:
# ── Regression benchmark: California Housing ──────────────────────────────────
cal = fetch_california_housing()
X_cal, y_cal = cal.data, cal.target
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_cal, y_cal, test_size=0.2, random_state=42)

reg_models = {
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'RF (100 trees)': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'RF (200 trees)': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                                   learning_rate=0.1, random_state=42),
}

rmse_results = {}
for name, model in reg_models.items():
    model.fit(X_tr_c, y_tr_c)
    pred = model.predict(X_te_c)
    rmse_results[name] = np.sqrt(mean_squared_error(y_te_c, pred))

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#95a5a6', '#3498db', '#2980b9', '#e74c3c']
bars = ax.bar(rmse_results.keys(), rmse_results.values(),
              color=colors, edgecolor='white')
ax.set_ylabel('RMSE (median house value, $100k units)')
ax.set_title('Regression RMSE — California Housing Dataset')
for bar, val in zip(bars, rmse_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

### 10.3 Hyperparameter Decision Guide

#### Random Forest

| Parameter | What it controls | Tuning direction |
|---|---|---|
| `n_estimators` | # trees | Increase until OOB error plateaus (200–500 usually sufficient) |
| `max_features` | De-correlation | Try `sqrt`, `log2`, `0.3`; smaller → less correlation, more bias |
| `max_depth` | Tree depth | Leave `None`; reduce if RAM is a concern |
| `min_samples_leaf` | Node size | Increase (2–5) for noisy data or regression |
| `max_samples` | Bootstrap fraction | Reduce (0.5–0.8) to further de-correlate trees |

#### Gradient Boosting

| Parameter | What it controls | Tuning direction |
|---|---|---|
| `learning_rate` | Shrinkage | Lower → better generalisation, needs more trees (0.01–0.1) |
| `n_estimators` | # stages | Coupled with learning rate; use early stopping |
| `max_depth` | Tree complexity | Keep shallow (3–5); boosting builds depth through stages |
| `subsample` | Row subsampling | 0.5–0.8 adds stochasticity, reduces overfitting |
| `min_samples_leaf` | Regularisation | Increase to 10–50 on large datasets |

### 10.4 When to Choose Which

| Situation | Recommended |
|---|---|
| Tabular data, need high accuracy | Gradient Boosting (XGBoost / LightGBM) |
| Need fast training + parallelism | Random Forest |
| Noisy data, outliers in labels | Random Forest (robust to label noise) |
| Small dataset | Random Forest (OOB avoids splitting data) |
| Interpretability needed | Single tree or RF + feature importance |
| Diverse model pool available | Stacking |
| Time series / causal ordering | Gradient Boosting with care around leakage |

---

## Summary

```
                   ENSEMBLE METHODS CONCEPTUAL MAP

  Single Learner     Problem            Ensemble Strategy
  ─────────────      ───────            ─────────────────
  Deep tree          High Variance   →  BAGGING      → Average parallel models
                                        RANDOM FOREST → Bagging + feature randomness

  Shallow tree       High Bias       →  BOOSTING     → Sequential error correction
  (weak learner)                        ADABOOST     → Re-weight misclassified
                                        GRAD BOOST   → Fit negative gradient

  Diverse models     Both            →  STACKING     → Meta-learner on OOF preds
```

**The three principles to internalise:**

1. **Averaging reduces variance** — but only to the extent that models are uncorrelated. Feature randomness in RF is specifically designed to push this limit.
2. **Sequential correction reduces bias** — boosting converts weak learners into a strong one by chaining corrections; the learning rate controls the step size of this gradient descent in function space.
3. **Diversity is the asset** — all ensemble gains trace back to learners being wrong in different ways. When learners agree perfectly, ensemble is no better than one model.

---

**References**

- Breiman, L. (1996). *Bagging predictors.* Machine Learning, 24(2), 123–140.
- Breiman, L. (2001). *Random forests.* Machine Learning, 45(1), 5–32.
- Freund, Y. & Schapire, R. (1997). *A decision-theoretic generalisation of on-line learning.* JCSS, 55(1), 119–139.
- Friedman, J. H. (2001). *Greedy function approximation: A gradient boosting machine.* Annals of Statistics, 29(5), 1189–1232.
- Geurts, P., Ernst, D. & Wehenkel, L. (2006). *Extremely randomized trees.* Machine Learning, 63, 3–42.
- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. Chapters 8, 10, 15.
- Louppe, G. et al. (2013). *Understanding variable importances in forests of randomized trees.* NeurIPS.